In [6]:
import sys
import time
import random
import string


class HashTable:
    """
    A hash table supporting Chaining and Quadratic Probing.
    Resizes (doubles capacity) when load factor exceeds 0.75.
    """

    def __init__(self, capacity: int = 16, method: str = "chaining"):
        """
        Args:
            capacity: Initial table size.
            method: 'chaining' or 'quadratic_probing'.
        """
        if method not in ("chaining", "quadratic_probing"):
            raise ValueError("method must be 'chaining' or 'quadratic_probing'")

        self.capacity = capacity
        self.method = method
        self.size = 0  
        self.load_factor_threshold = 0.75

       
        if method == "chaining":
            self.table = [[] for _ in range(capacity)]
        else:
            self.keys = [None] * capacity
            self.values = [None] * capacity

        self.collision_count = 0
        self.insert_count = 0
        self.get_count = 0
        self.delete_count = 0

    def hash_function(self, key) -> int:
        """Returns the base hash index for *key*."""
        return hash(key) % self.capacity

    def _resize(self):
        """Double the table capacity and rehash every existing element."""
        old_capacity = self.capacity
        old_items = []

        if self.method == "chaining":
            for bucket in self.table:
                for k, v in bucket:
                    old_items.append((k, v))
        else:
            for k, v in zip(self.keys, self.values):
                if k is not None:
                    old_items.append((k, v))

        self.capacity *= 2
        self.size = 0  

        if self.method == "chaining":
            self.table = [[] for _ in range(self.capacity)]
        else:
            self.keys = [None] * self.capacity
            self.values = [None] * self.capacity

        for k, v in old_items:
            self._insert_internal(k, v)

    def _should_resize(self) -> bool:
        return self.size / self.capacity >= self.load_factor_threshold

    def _insert_internal(self, key, value):
        """Insert without updating collision or operation counters."""
        if self.method == "chaining":
            idx = self.hash_function(key)
            bucket = self.table[idx]
            for i, (k, _) in enumerate(bucket):
                if k == key:
                    bucket[i] = (key, value)
                    return
            bucket.append((key, value))
        else:
            idx = self.hash_function(key)
            for i in range(self.capacity):
                probe = (idx + i * i) % self.capacity
                if self.keys[probe] is None:
                    self.keys[probe] = key
                    self.values[probe] = value
                    return
                if self.keys[probe] == key:
                    self.values[probe] = value
                    return

    def insert(self, key, value):
        """Insert or update a key-value pair."""
        if self._should_resize():
            self._resize()

        self.insert_count += 1

        if self.method == "chaining":
            idx = self.hash_function(key)
            bucket = self.table[idx]
            for k, _ in bucket:
                if k == key:
                    for i, (k, _) in enumerate(bucket):
                        if k == key:
                            bucket[i] = (key, value)
                            return
            if len(bucket) > 0:
                self.collision_count += 1
            bucket.append((key, value))
            self.size += 1

        else: 
            idx = self.hash_function(key)
            found = False
            for i in range(self.capacity):
                probe = (idx + i * i) % self.capacity
                if self.keys[probe] is None:
                    self.keys[probe] = key
                    self.values[probe] = value
                    self.size += 1
                    if i > 0:
                        self.collision_count += 1
                    return
                if self.keys[probe] == key:
                    self.values[probe] = value
                    if i > 0:
                        self.collision_count += 1
                    return
                self.collision_count += 1
            raise RuntimeError("Hash table is full; cannot insert.")

    def get(self, key):
        """Return the value for *key*, or None if not found."""
        self.get_count += 1

        if self.method == "chaining":
            idx = self.hash_function(key)
            for k, v in self.table[idx]:
                if k == key:
                    return v
            return None

        else: 
            idx = self.hash_function(key)
            for i in range(self.capacity):
                probe = (idx + i * i) % self.capacity
                if self.keys[probe] is None:
                    return None                
                if self.keys[probe] == key:
                    return self.values[probe]
            return None  

    def delete(self, key):
        """Delete the key-value pair for *key*. Returns True if found."""
        self.delete_count += 1

        if self.method == "chaining":
            idx = self.hash_function(key)
            bucket = self.table[idx]
            for i, (k, _) in enumerate(bucket):
                if k == key:
                    bucket.pop(i)
                    self.size -= 1
                    return True
            return False

        else:  
            idx = self.hash_function(key)
            for i in range(self.capacity):
                probe = (idx + i * i) % self.capacity
                if self.keys[probe] is None:
                    return False  
                if self.keys[probe] == key:
                    self.keys[probe] = None
                    self.values[probe] = None
                    self.size -= 1
                    return True
            return False

    @property
    def load_factor(self) -> float:
        return self.size / self.capacity

    def get_collision_count(self) -> int:
        return self.collision_count

    def get_memory_usage(self) -> int:
        """Rough memory estimate in bytes (via sys.getsizeof)."""
        if self.method == "chaining":
            total = sys.getsizeof(self.table)
            for bucket in self.table:
                total += sys.getsizeof(bucket)
                for item in bucket:
                    total += sys.getsizeof(item)
            return total
        else:
            return (sys.getsizeof(self.keys) +
                    sys.getsizeof(self.values))

    def __repr__(self):
        return (f"HashTable(capacity={self.capacity}, size={self.size}, "
                f"method='{self.method}', collisions={self.collision_count})")


def _benchmark_insert(method: str, num_items: int, seed: int = 42) -> dict:
    """Insert *num_items* random strings and return timing / stats."""
    rng = random.Random(seed)
    ht = HashTable(capacity=4, method=method)  

    start = time.perf_counter()
    for _ in range(num_items):
        key = f"key_{rng.randint(0, num_items * 10)}"
        value = rng.randint(0, 100000)
        ht.insert(key, value)
    elapsed = time.perf_counter() - start

    return {
        "method": method,
        "num_items": num_items,
        "final_capacity": ht.capacity,
        "load_factor": round(ht.load_factor, 4),
        "collisions": ht.collision_count,
        "insert_time_sec": round(elapsed, 6),
        "memory_bytes": ht.get_memory_usage(),
        "insert_count": ht.insert_count,
    }

# Test for Chaining
hash_table_chaining = HashTable(10, method="chaining")
hash_table_chaining.insert("apple", 5)
hash_table_chaining.insert("banana", 7)
hash_table_chaining.insert("cherry", 3)
print("Chaining - Get apple:", hash_table_chaining.get("apple"))  # Should return 5
print("Chaining - Get banana:", hash_table_chaining.get("banana"))  # Should return 7
hash_table_chaining.delete("apple")
print("Chaining - Get apple after deletion:", hash_table_chaining.get("apple"))  # Should return None

# Test for Quadratic Probing
hash_table_quadratic = HashTable(10, method="quadratic_probing")
hash_table_quadratic.insert("apple", 5)
hash_table_quadratic.insert("banana", 7)
hash_table_quadratic.insert("cherry", 3)
print("Quadratic Probing - Get apple:", hash_table_quadratic.get("apple"))  # Should return 5
print("Quadratic Probing - Get banana:", hash_table_quadratic.get("banana"))  # Should return 7
hash_table_quadratic.delete("apple")
print("Quadratic Probing - Get apple after deletion:", hash_table_quadratic.get("apple"))  # Should return None


def _benchmark_mixed_operations(method: str, num_items: int, seed: int = 42) -> dict:
    """Insert, then mix inserts / gets / deletes and measure."""
    rng = random.Random(seed)
    ht = HashTable(capacity=4, method=method)

    keys = []
    for _ in range(num_items):
        key = f"key_{rng.randint(0, num_items * 5)}"
        value = rng.randint(0, 100000)
        ht.insert(key, value)
        keys.append(key)

    ops = 0
    hits = 0
    deletes = 0
    start = time.perf_counter()
    for _ in range(num_items):
        op = rng.choice(["get", "get", "get", "insert", "delete"])
        key = rng.choice(keys)
        if op == "get":
            _ = ht.get(key)
            hits += 1
        elif op == "insert":
            new_key = f"new_{rng.randint(0, num_items * 10)}"
            ht.insert(new_key, rng.randint(0, 100000))
        else:
            if ht.delete(key):
                deletes += 1
        ops += 1
    elapsed = time.perf_counter() - start

    return {
        "method": method,
        "num_items": num_items,
        "final_capacity": ht.capacity,
        "load_factor": round(ht.load_factor, 4),
        "collisions": ht.collision_count,
        "mixed_ops_time_sec": round(elapsed, 6),
        "memory_bytes": ht.get_memory_usage(),
        "total_ops": ops,
        "hits": hits,
        "deletes": deletes,
    }


def run_analysis():
    """Run a suite of benchmarks and print a comparison table."""
    sizes = [1_000, 10_000, 50_000]

    print("=" * 90)
    print("  HASH TABLE PERFORMANCE ANALYSIS")
    print("  Comparing: Chaining  vs  Quadratic Probing")
    print("=" * 90)

    print("\n--- Insert-Only Benchmark ---")
    print(f"{'Size':>8} | {'method':<22} | {'Collisions':>12} | "
          f"{'Insert Time (s)':>16} | {'Final Cap':>10} | {'Memory (B)':>10}")
    print("-" * 90)

    for n in sizes:
        for tech in ("chaining", "quadratic_probing"):
            res = _benchmark_insert(tech, n, seed=42)
            print(f"{res['num_items']:>8,} | {res['method']:<22} | "
                  f"{res['collisions']:>12,} | {res['insert_time_sec']:>16.6f} | "
                  f"{res['final_capacity']:>10,} | {res['memory_bytes']:>10,}")

    print("\n--- Mixed Insert / Get / Delete Benchmark ---")
    print(f"{'Size':>8} | {'method':<22} | {'Collisions':>12} | "
          f"{'Mixed Ops Time (s)':>18} | {'Memory (B)':>10} | {'Hits':>8}")
    print("-" * 90)

    for n in sizes:
        for tech in ("chaining", "quadratic_probing"):
            res = _benchmark_mixed_operations(tech, n, seed=42)
            print(f"{res['num_items']:>8,} | {res['method']:<22} | "
                  f"{res['collisions']:>12,} | {res['mixed_ops_time_sec']:>18.6f} | "
                  f"{res['memory_bytes']:>10,} | {res['hits']:>8,}")


Chaining - Get apple: 5
Chaining - Get banana: 7
Chaining - Get apple after deletion: None
Quadratic Probing - Get apple: 5
Quadratic Probing - Get banana: 7
Quadratic Probing - Get apple after deletion: None
